In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/achiver7/project-phase-1/data/sc_val.jsonl
/kaggle/input/datasets/achiver7/project-phase-1/data/acts_val.jsonl
/kaggle/input/datasets/achiver7/project-phase-1/data/acts_train.jsonl
/kaggle/input/datasets/achiver7/project-phase-1/data/tribunals_val.jsonl
/kaggle/input/datasets/achiver7/project-phase-1/data/test.json
/kaggle/input/datasets/achiver7/project-phase-1/data/sc_train.jsonl
/kaggle/input/datasets/achiver7/project-phase-1/data/tribunals_train.jsonl


In [2]:
# ============== TOKENIZER PHASE ONLY: train tokenizer -> pack -> verify ==============
import os, json, re, time, shutil, itertools
import numpy as np
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

D   = "/kaggle/input/datasets/achiver7/project-phase-1/data"   # raw *.jsonl + test.json
OUT = "/kaggle/working/tokens"; os.makedirs(OUT, exist_ok=True)
VOCAB_SIZE, DOCS_PER_SOURCE = 40000, 3000
DROP_JUNK, STRIP_DEVANAGARI = True, True                       # tribunals only

# ---- 1. tokenizer ----
tok_path = f"{OUT}/tokenizer.json"
if not os.path.exists(tok_path):
    def text_iter():
        for name in ["acts_train", "sc_train", "tribunals_train"]:
            with open(f"{D}/{name}.jsonl", encoding="utf-8") as f:
                for line in itertools.islice(f, DOCS_PER_SOURCE):
                    yield json.loads(line)["text"]
    t = Tokenizer(models.BPE())
    t.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    t.decoder = decoders.ByteLevel()
    t.train_from_iterator(text_iter(), trainers.BpeTrainer(
        vocab_size=VOCAB_SIZE, special_tokens=["<pad>", "<bos>", "<eos>"],
        initial_alphabet=pre_tokenizers.ByteLevel.alphabet()))
    t.save(tok_path)
tok = Tokenizer.from_file(tok_path); EOS = tok.token_to_id("<eos>")
assert tok.get_vocab_size() == VOCAB_SIZE and EOS == 2 and VOCAB_SIZE < 65536
print(f"[1] tokenizer OK: vocab {tok.get_vocab_size()}, eos id {EOS}")

# ---- 2. pack ----
dev_re, c1_re = re.compile(r"[\u0900-\u097F]+"), re.compile(r"[\x80-\x9f]")
def clean_tribunal(t):
    n = max(len(t), 1)
    if DROP_JUNK and (len(c1_re.findall(t)) / n > .001 or t.count("?") / n > .05): return None
    if STRIP_DEVANAGARI: t = " ".join(dev_re.sub(" ", t).split())
    return t or None

def pack(src, dst, clean=None, batch=512):
    out = f"{OUT}/{dst}.bin"
    if os.path.exists(out): print(f"   skip {dst} (exists)"); return
    t0, arrs, buf, kept = time.time(), [], [], 0
    def flush():
        for e in tok.encode_batch(buf): arrs.append(np.array(e.ids + [EOS], dtype=np.uint16))
        buf.clear()
    with open(f"{D}/{src}.jsonl", encoding="utf-8") as f:
        for line in f:
            t = json.loads(line)["text"]
            if clean: t = clean(t)
            if t: buf.append(t); kept += 1
            if len(buf) == batch: flush()
    if buf: flush()
    ids = np.concatenate(arrs); ids.tofile(out)
    print(f"   {dst}: {len(ids)/1e6:.1f}M tokens, {kept} docs, {time.time()-t0:.0f}s")

for src, dst, cl in [("acts_val", "acts_val", None), ("sc_val", "sc_val", None),
                     ("tribunals_val", "tribunals_val_f", clean_tribunal),
                     ("acts_train", "acts_train", None), ("sc_train", "sc_train", None),
                     ("tribunals_train", "tribunals_train_f", clean_tribunal)]:
    pack(src, dst, cl)
if not os.path.exists(f"{OUT}/test.json"): shutil.copy(f"{D}/test.json", f"{OUT}/test.json")
print("[2] packing OK")

# ---- 3. verify ----
dev_chk = re.compile(r"[\u0900-\u097F]")
for n in ["acts_train", "sc_train", "tribunals_train_f", "acts_val", "sc_val", "tribunals_val_f"]:
    d = np.memmap(f"{OUT}/{n}.bin", dtype=np.uint16, mode="r")
    assert int(d.max()) < VOCAB_SIZE, f"{n}: token id out of range"
    mid = len(d) // 2
    sample = tok.decode(d[mid:mid + 60].tolist())
    left = len(dev_chk.findall(tok.decode(d[mid:mid + 50000].tolist()))) if "tribunals" in n else 0
    print(f"   {n:18s} {len(d)/1e6:7.1f}M tokens | max id {int(d.max())} | Devanagari left {left}\n      {sample[:160]!r}")
test = json.load(open(f"{OUT}/test.json"))
ln = [len(e.ids) for e in tok.encode_batch([v["text"] for v in test.values()])]
print(f"[3] verify OK | test prompts: {len(test)} | tokens mean {sum(ln)/len(ln):.0f}, max {max(ln)}")
print("TOKENIZER PHASE COMPLETE:", sorted(os.listdir(OUT)))




[1] tokenizer OK: vocab 40000, eos id 2
   acts_val: 3.4M tokens, 367 docs, 5s
   sc_val: 4.8M tokens, 1085 docs, 6s
   tribunals_val_f: 28.4M tokens, 17125 docs, 35s
   acts_train: 145.6M tokens, 18455 docs, 196s
   sc_train: 249.8M tokens, 54443 docs, 310s
   tribunals_train_f: 137.6M tokens, 85052 docs, 170s
[2] packing OK
   acts_train           145.6M tokens | max id 39999 | Devanagari left 0
      ' in sub-section (2), for the words "a President Vice-President, or member" the words "a President or Vice-President" shall he substituted; (3) in sub-section (3'
   sc_train             249.8M tokens | max id 39999 | Devanagari left 0
      ' the contention raised by the respondents. The respondents then filed a petition in the High Court of Bombay under Art. 226 of the Constitution praying that the'
   tribunals_train_f    137.6M tokens | max id 39999 | Devanagari left 0
      ' presumed that the said spark caused the fire, it can not be accepted that it spread so immediately to ot